In [41]:
import kagglehub
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util

In [42]:
print("Скачиваем датасет TMDB...")
path = kagglehub.dataset_download("tmdb/tmdb-movie-metadata")

print("Путь:", path)
print("Файлы:", os.listdir(path))

movies_path = os.path.join(path, "tmdb_5000_movies.csv")
df = pd.read_csv(movies_path)

print(f"\nУспешно загружено {len(df)} строк.")

Скачиваем датасет TMDB...


ReadTimeout: HTTPSConnectionPool(host='www.kaggle.com', port=443): Read timed out. (read timeout=5)

In [ ]:
df_simple = df[['title', 'overview', 'release_date', 'runtime']].copy()

# Очищаем от пустых описаний
df_simple = df_simple.dropna(subset=['overview'])
df_simple = df_simple[df_simple['overview'].str.strip() != '']

# Извлекаем год
df_simple['release_date'] = pd.to_datetime(df_simple['release_date'], errors='coerce')
df_simple['year'] = df_simple['release_date'].dt.year.fillna(0).astype(int)

# Сбрасываем индекс
df_simple.reset_index(drop=True, inplace=True)

df_simple.to_csv('movies_simple.csv', index=False)
print(f"Сохранено {len(df_simple)} фильмов.")

texts = df_simple['overview'].tolist()
titles = df_simple['title'].tolist()

Сохранено 4799 фильмов.


In [ ]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

In [ ]:
embeddings = model.encode(texts, show_progress_bar=True)
np.save('movie_embeddings.npy', embeddings)

Batches:   0%|          | 0/150 [00:00<?, ?it/s]

In [ ]:
def search(
    query: str,
    model,
    embeddings: np.ndarray,
    df: pd.DataFrame,
    top_k: int = 10,
    year_from: int = None,
    year_to: int = None,
    min_similarity: float = 0.1
) -> pd.DataFrame:
    """
    Выполняет семантический поиск фильмов по текстовому запросу.

    Args:
        query: текстовый запрос пользователя (на любом языке)
        model: экземпляр SentenceTransformer
        embeddings: матрица эмбеддингов описаний фильмов (shape: N x D)
        df: DataFrame с колонками ['title', 'overview', 'year']
        top_k: количество возвращаемых результатов
        year_from, year_to: фильтрация по году выпуска
        min_similarity: минимальный порог схожести (фильтр шума)

    Returns:
        DataFrame с колонками: title, overview, year, similarity
    """
    if not query.strip():
        return pd.DataFrame()

    # 1. Применяем фильтрацию по году (если задана)
    mask = pd.Series([True] * len(df))
    if year_from is not None:
        mask &= (df['year'] >= year_from)
    if year_to is not None:
        mask &= (df['year'] <= year_to)

    filtered_df = df[mask].copy()
    if filtered_df.empty:
        return pd.DataFrame()

    # 2. Получаем индексы и подмножество эмбеддингов
    indices = filtered_df.index.tolist()
    filtered_embeddings = embeddings[indices]

    # 3. Эмбеддинг запроса
    query_emb = model.encode(query, convert_to_tensor=False, show_progress_bar=False)

    # 4. Считаем косинусное сходство (с использованием Sentence-Transformers утилиты)
    similarities = util.cos_sim(query_emb, filtered_embeddings)[0].cpu().numpy()

    # 5. Применяем порог и сортируем
    valid_mask = similarities >= min_similarity
    if not valid_mask.any():
        return pd.DataFrame()

    top_k = min(top_k, valid_mask.sum())
    top_indices_local = np.argsort(similarities)[::-1][:top_k]

    # 6. Формируем результат
    results = []
    for i in top_indices_local:
        if similarities[i] < min_similarity:
            continue
        orig_idx = indices[i]
        results.append({
            'title': df.loc[orig_idx, 'title'],
            'overview': df.loc[orig_idx, 'overview'],
            'year': df.loc[orig_idx, 'year'],
            'similarity': float(similarities[i])
        })

    return pd.DataFrame(results)

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer, util

# ============================================================================
# Настройка страницы
# ============================================================================
st.set_page_config(
    page_title="🎬 Нейропоиск фильмов",
    page_icon="🧠",
    layout="centered"
)

st.title("🧠 Нейросетевой поиск фильмов по смыслу")
st.markdown(
    "Опишите сюжет — мы найдём подходящие фильмы, даже если вы не знаете названия. "
    "Поддерживается русский и английский языки."
)

# ============================================================================
# Загрузка данных и модели
# ============================================================================
@st.cache_resource
def load_model():
    return SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

@st.cache_data
def load_data_and_embeddings():
    if not os.path.exists("movies_simple.csv"):
        st.error("❌ Файл 'movies_simple.csv' не найден. Подготовьте данные.")
        st.stop()
    if not os.path.exists("movie_embeddings.npy"):
        st.error("❌ Файл 'movie_embeddings.npy' не найден. Создайте эмбеддинги.")
        st.stop()

    df = pd.read_csv("movies_simple.csv")
    embeddings = np.load("movie_embeddings.npy")
    return df, embeddings

try:
    model = load_model()
    df, embeddings = load_data_and_embeddings()
except Exception as e:
    st.error(f"❌ Ошибка при загрузке: {e}")
    st.stop()

# ============================================================================
# Функция нейронного поиска
# ============================================================================
def neural_search(query, df, embeddings, model, top_k=10, year_from=1900, year_to=2025, min_sim=0.1):
    if not query.strip():
        return pd.DataFrame()

    # Фильтрация по году
    mask = (df['year'] >= year_from) & (df['year'] <= year_to)
    filtered_df = df[mask]
    if filtered_df.empty:
        return pd.DataFrame()

    indices = filtered_df.index.tolist()
    filtered_embs = embeddings[indices]

    # Эмбеддинг запроса
    with st.spinner("Анализируем запрос..."):
        query_emb = model.encode(query, convert_to_tensor=True)
        sims = util.cos_sim(query_emb, filtered_embs)[0].cpu().numpy()

    # Сортировка и отбор
    top_idx = np.argsort(sims)[::-1]
    results = []
    for i in top_idx:
        if sims[i] < min_sim or len(results) >= top_k:
            break
        orig_idx = indices[i]
        year = df.loc[orig_idx, 'year']
        year_display = int(year) if pd.notna(year) and year > 0 else "???"
        results.append({
            'title': df.loc[orig_idx, 'title'],
            'overview': df.loc[orig_idx, 'overview'],
            'year': year_display,
            'similarity': float(sims[i])
        })
    return pd.DataFrame(results)

# ============================================================================
# Интерфейс пользователя
# ============================================================================
query = st.text_area(
    "🔍 Описание фильма",
    placeholder="Например: «девушка теряет память после аварии, но её преследуют сны о космосе»",
    height=100
)

col1, col2 = st.columns(2)
with col1:
    year_from = st.number_input("Год от", min_value=1900, max_value=2025, value=1900)
with col2:
    year_to = st.number_input("Год до", min_value=1900, max_value=2025, value=2025)

if st.button("🎬 Найти фильмы"):
    if not query.strip():
        st.warning("⚠️ Пожалуйста, введите описание фильма.")
    else:
        with st.spinner("Ищем подходящие фильмы..."):
            results = neural_search(
                query=query,
                df=df,
                embeddings=embeddings,
                model=model,
                top_k=8,
                year_from=year_from,
                year_to=year_to,
                min_sim=0.1
            )

        if results.empty:
            st.info("📭 Ничего не найдено. Попробуйте изменить запрос или расширить диапазон лет.")
        else:
            st.subheader(f"✅ Найдено {len(results)} фильмов")
            for _, r in results.iterrows():
                st.markdown(f"### 🎥 {r['title']} ({r['year']})")
                st.markdown(f"**Семантическая схожесть**: `{r['similarity']:.3f}`")
                st.write(r['overview'])
                st.markdown("---")

# ============================================================================
# Примеры запросов (аналогично eLIBRARY)
# ============================================================================
with st.expander("💡 Примеры эффективных запросов (как в eLIBRARY)"):
    st.write("""
    - *агент 007 расследует заговор, связанный с ИИ*
    - *любовь между человеком и роботом в Токио будущего*
    - *пираты находят карту сокровищ на затонувшем корабле*
    - *женщина получает способность читать мысли и раскаивается*
    - *выжившие после апокалипсиса ищут чистую воду в пустыне*
    """)

Overwriting app.py
